In [2]:
# viz.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind
import warnings
warnings.filterwarnings('ignore')

In [3]:
def set_style():
    # 한글 폰트
    try:
        import koreanize_matplotlib  # noqa
    except Exception as e:
        return f"에러가 발생했음: {e}"


    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 200,
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

In [4]:
# boxplt_with_mean

def boxplt_with_mean(df, x, y, order, title, outlier_rule=None): # df=DataFrame, x=str, y=str, order=list, title=str, outlier_rule 예: {"value": 4.0, "label": "시력 이상치 기준(>=4.0)"}
    fig, ax = plt.subplots(figsize=(10, 5))

    # --------------------------------
    # 1) boxplot 설정
    # --------------------------------

    palette_range = (0.20, 0.90)
    cmap = sns.color_palette("blend:#7AB,#EDA", as_cmap=True)
    palette = [cmap(t) for t in np.linspace(*palette_range, len(order))]

    sns.boxplot(
        data = df, x = x, y = y, order = order,
        width = 0.55, fliersize = 3, linewidth = 1,
        ax = ax, palette = palette,
        medianprops = {"color": "#B3B3B3", "linewidth": 1.5},
        notch = True
    )


    # --------------------------------
    # 2) 평균 삽입
    # --------------------------------
    # 2-1) 평균 계산
    means = (df.groupby(x)[y].mean().reindex(order))

    # 2-2) 평균(다이아몬드) + 평균선(짧게) + 주석(절제)
    for i, m in enumerate(means):
        if pd.isna(m):
            continue

        # 평균 마커(다이아몬드)
        ax.scatter(i, m, marker="D", s=70, color="#2F2F2F", alpha=0.75, zorder=5)
        # 평균 주석(오른쪽 위로 살짝 이동. 곂침 최소화)
        ax.annotate(
            f"평균: {round(m, 2)}",
            xy=(i, m),
            xytext=(12, 8),             # (x,y) 픽셀 단위 이동 → 겹침 방지 핵심
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=10,
            color="#2F2F2F",
            bbox=dict(
                boxstyle="round,pad=0.25",
                fc="white",
                ec="#2F2F2F",
                lw=0.8,
                alpha=0.95
            )
        )

    # --------------------------------
    # 3) 이상치 표시
    # --------------------------------

    if outlier_rule is not None and "value" in outlier_rule:
        v = outlier_rule["value"]
        label = outlier_rule.get("label", f"기준: {v}")

        ax.axhline(v, linestyle="--", linewidth=1.2, alpha=0.7, color="#FFB6C1")

        # ✅ axes 좌표로 고정
        ax.text(
            0.98, 0.92,   # 오른쪽 상단
            label,
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=10,
            color="#2F2F2F",
            clip_on=False
        )

    # --------------------------------
    # 4) 미감 정리 (제목 Bold, 그리드 은은)
    # --------------------------------

    ax.set_title(f"{title} (Boxplot + Mean)", pad=12, fontweight="bold")
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.grid(axis="y", linestyle=":", alpha=0.35)
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.show()


In [ ]:
# histogram

def hist_one_feature(df, x, title, bins=30, upper=None):

    # --------------------------------
    # 1) histplot 설정
    # --------------------------------

    plt.figure(figsize=(8, 4))
    ax = sns.histplot(
        data = df,
        x = x,
        bins=bins,
        color = "#ADD8E6",
        kde = True
    )

    # --------------------------------
    # 2) 평균 삽입
    # --------------------------------

    mean = df[x].mean()

    # 2-1) 평균선

    ax.axvline(
        mean,
        color="#C44E52",
        linestyle="--",
        linewidth=1.5,
        alpha=0.8
    )

    # 2-2) 평균 라벨 박스
    ax.annotate(
        f"평균: {mean:.2f}",
        xy=(mean, ax.get_ylim()[1]),
        xytext=(6, -6),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round,pad=0.25",
            fc="white",
            ec="#C44E52",
            lw=0.8
        )
    )

    # --------------------------------
    # 3) 중앙값 삽입
    # --------------------------------

    median = df[x].median()

    # 중앙값

    ax.axvline(median, color="#DD8452", linestyle="-.", linewidth=1.2)

    ax.annotate(
        f"중앙값: {median:.2f}",
        xy=(median, ax.get_ylim()[1]),
        xytext=(6, -22),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round,pad=0.25",
            fc="white",
            ec="#DD8452",
            lw=0.8
        )
    )

    # --------------------------------
    # 4) 🔴 극단적 이상치 영역 박스
    # --------------------------------

    # 정상 범주
    ax.axvspan(
        upper,
        df[x].max(),
        color="#D55E00",
        alpha=0.12
    )

    ax.text(
        upper + 5,
        ax.get_ylim()[1] * 0.85,
        "극단적 고혈당 영역\n(의학적 이상치)",
        fontsize=10,
        color="#7A2E00",
        ha="left",
        va="top"
    )

    # --------------------------------
    # 4) 미감 정리
    # --------------------------------

    ax.set_title(title, fontweight="bold", pad=10)
    ax.set_xlabel(x)
    ax.set_ylabel("Count")
    ax.grid(axis="y", linestyle=":", alpha=0.3)
    sns.despine()

    plt.tight_layout()
    plt.show()

In [ ]:
# 전체 feature 특성 한번에 파악(로그 변환 필요 유무 파악)
import math

def all_feature_plot(df, feature_list):

    feature_list = feature_list
    n_cols = 4                              # 한 줄에 4개씩 (가독성 좋음)
    n_rows = math.ceil(len(feature_list) / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4 * n_cols, 3 * n_rows)
    )

    axes = axes.flatten()

    # --------------------------------
    # 1) histplot 설정
    # --------------------------------

    cmap = sns.color_palette("blend:#7AB,#EDA", as_cmap=True)
    colors = [cmap(t) for t in np.linspace(0.2, 0.9, len(feature_list))]

    for ax, col, color in zip(axes, feature_list, colors):
        # 1-1) 히스토그램 + KDE
        sns.histplot(
            data=df,
            x=col,
            stat="density",       # KDE와 스케일 맞춤
            kde=True,
            color=color,
            alpha=0.55,
            edgecolor="#333333",
            ax=ax
        )

    # --------------------------------
    # 2) 왜도 정보 추가
    # --------------------------------

        # 2-1) 왜도 계산
        skew = df[col].skew()

        # 2-2) 제목에 feature명 + 왜도 숫자 표시
        title_color = "#F08080" if abs(skew) > 1.5 else "#2F2F2F"
        ax.set_title(
            f"{col}\nskew={skew:.2f}",
            fontsize=9,
            fontweight="bold",
            color=title_color
        )

    # --------------------------------
    # 3) 미감 정리
    # --------------------------------

    ax.grid(axis="y", linestyle=":", alpha=0.25)
    sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("#333333")

    # 3-2) 남는 빈 subplot 제거
    for i in range(len(feature_list), len(axes)):
        fig.delaxes(axes[i])

    plt.tight_layout()
    plt.show()

In [ ]:
# 범주 vs. 범주 countplot

SMOKER_PALETTE = {"흡연자": "#F08080", "비흡연자": "#B0E0E6"}

def countplot_feature_smoke(
    df,
    x,
    title,
    hue="Smoker",
    order=None,
    palette=SMOKER_PALETTE
):
    d = df.copy()

    # -----------------------------
    # 1) countplot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.countplot(
        data=d,
        x=x,
        hue=hue,
        hue_order=list(palette.keys()),   # ✅ 추가: hue 순서 강제
        order=order,
        palette=palette,
        ax=ax,
    )

    # -----------------------------
    # 2) 막대 위 비율 라벨
    # -----------------------------
    hue_order = ["흡연자", "비흡연자"]  # 네가 원하는 표시 순서로 고정

    rate = (
        pd.crosstab(df[x], df[hue], normalize="index") * 100
        ).reindex(index=order, columns=hue_order)
    for j, container in enumerate(ax.containers):   # j=0이면 흡연자, j=1이면 비흡연자
        smoker_label = hue_order[j]

        for bar in container:
            h = bar.get_height()
            if h == 0:
                continue

            # 이 막대가 속한 x(연령대) 인덱스 구하기
            x_center = bar.get_x() + bar.get_width()/2
            x_idx = int(round(x_center))
            age_label = ax.get_xticklabels()[x_idx].get_text()

            pct = rate.loc[age_label, smoker_label]  # ✅ 컬럼명으로 접근 (순서 영향 X)

            ax.annotate(
                f"{pct:.1f}%",
                (bar.get_x() + bar.get_width()/2, h),
                ha="center", va="bottom",
                xytext=(0, 4), textcoords="offset points",
                fontsize=10,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#999", alpha=0.9),
                clip_on=False,
            )

    # -----------------------------
    # 3) 스타일
    # -----------------------------
    ax.set_title(title, fontweight="bold", pad=10)
    ax.set_xlabel(x)
    ax.set_ylabel("Count")
    ax.grid(axis="y", linestyle=":", alpha=0.3)
    sns.despine(ax=ax, top=True, right=True)
    ax.legend(title=hue, frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
# 범주 vs. 연속형 histplot with hue
SMOKER_PALETTE = {"흡연자": "#F08080", "비흡연자": "#B0E0E6"}

def hist_with_hue(df, x, title, hue=None, bins=30):

    # 그룹 순서: 팔레트 키 순서와 일치시키면 모든 그래프가 안정적
    hue_order = list(SMOKER_PALETTE.keys())

    # -------------------------
    # 1) Histogram + KDE 설정
    # -------------------------
    plt.figure(figsize=(8, 4))

    ax = sns.histplot(
        data=df,
        x=x,
        hue=hue,
        hue_order=hue_order,
        bins=bins,
        kde=True,
        stat="count",          # count로 유지 (원하면 density로 변경 가능)
        common_norm=False,     # 그룹별 분포 비교 시 유리
        palette=SMOKER_PALETTE,
        alpha=0.45,
        edgecolor="#333333",
        linewidth=0.6
    )

    # 그룹별 평균/중앙값 라인 + 라벨
    # 평균/중앙값 계산
    stats = (
        df
        .groupby(hue)[x]
        .agg(mean="mean", median="median")
    )

    ax = plt.gca()
    y_max = ax.get_ylim()[1]

    for smoker, row in stats.iterrows():
        mean_v = row["mean"]
        med_v = row["median"]
        color = SMOKER_PALETTE[smoker]

        # 세로선 (중앙값/평균)
        ax.axvline(mean_v, color="#C44E52", linestyle="--", linewidth=2, alpha=0.9)
        ax.axvline(med_v,  color="#DD8452", linestyle="-.", linewidth=2, alpha=0.9)

        # 텍스트 위치 분리
        if smoker == "흡연자":
            x_text = mean_v + 0.06   # 👉 오른쪽
            align = "left"
            y_text = y_max * 0.92
        else:
            x_text = mean_v - 0.06   # 👈 왼쪽
            align = "right"
            y_text = y_max * 0.80

        ax.text(
            x_text,
            y_text,
            f"{smoker}\n평균 {mean_v:.2f}\n중앙값 {med_v:.2f}",
            ha=align,
            va="top",
            fontsize=10,
            bbox=dict(
                boxstyle="round,pad=0.3",
                fc="white",
                ec=color,
                alpha=0.9
            )
        )

    # 선 범례 추가

    # 흡연자/비흡연자 범례를 직접 생성 (seaborn 의존 X)
    handles_smoker = [
        Patch(facecolor=SMOKER_PALETTE["흡연자"], edgecolor="#F08080", alpha=0.45, label="흡연자"),
        Patch(facecolor=SMOKER_PALETTE["비흡연자"], edgecolor="#B0E0E6", alpha=0.45, label="비흡연자"),
    ]

    # 통계 지표 범례
    handles_stats = [
        Line2D([0], [0], color="#C44E52", lw=2, linestyle="--", label="평균"),
        Line2D([0], [0], color="#DD8452", lw=2, linestyle="-.", label="중앙값"),
    ]

    ax.legend(handles=handles_smoker + handles_stats, title="범례", frameon=True, loc="upper right")


    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(x)
    ax.set_ylabel("Count")
    ax.grid(axis="y", linestyle=":", alpha=0.25)
    sns.despine(ax=ax)

In [ ]:
# 범주 vs. 연속 barplot with p-value

    # -------------------------
    # 0) p-value 표기 방법 정의
    # -------------------------

def star_from_p(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

SMOKER_PALETTE = {"흡연자": "#F08080", "비흡연자": "#B0E0E6"}

    # -------------------------
    # 1) 막대 그래프 설정
    # -------------------------

def bar_with_hue(df, x, y, title, y_label, hue=None, order=None):

    # (1) 복사본에서만 처리 (원본 df 건드리지 않기)
    d = df.copy()

    # hue 순서 고정
    hue_order = list(SMOKER_PALETTE.keys())  # ["흡연자","비흡연자"]

    # x 순서 고정
    if order is not None:
        d[x] = pd.Categorical(d[x], categories=order, ordered=True)

    # (2) 요약통계: mean/sd/n
    summary = (
        d.dropna(subset=[x, y, hue])
         .groupby([x, hue])[y]
         .agg(mean="mean", sd="std", n="count")
         .reset_index()
    )
    summary["sd"] = summary["sd"].fillna(0)

    # (3) 막대는 seaborn으로(에러바 OFF)
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(
        data=d,
        x=x,
        y=y,
        hue=hue,
        order=order,
        hue_order=hue_order if hue is not None else None,
        estimator=np.mean,
        ci=None,                 # 구버전 호환: 에러바 OFF
        palette=SMOKER_PALETTE,
        ax=ax
    )

    # (4) bar container 안전하게 잡기 (막대 컨테이너만)
    bar_containers = [c for c in ax.containers if hasattr(c, "patches")]
    bar_containers = bar_containers[:len(hue_order)]  # 2개만(흡연/비흡연)

    capsize = 7

    # 오차막대 + 평균 텍스트
    for h_i, container in enumerate(bar_containers):
        h_name = hue_order[h_i]

        for x_i, bar in enumerate(container.patches):
            x_label = ax.get_xticklabels()[x_i].get_text()

            row = summary[(summary[x] == x_label) & (summary[hue] == h_name)]
            if row.empty:
                continue

            mean_v = float(row["mean"].iloc[0])
            sd_v   = float(row["sd"].iloc[0])

            x_center = bar.get_x() + bar.get_width() / 2

            # ✅ 오차막대: mean_v 기준으로 강제 (매칭 안정화)
            ax.errorbar(
                x=x_center, y=mean_v,
                yerr=sd_v,
                fmt="none",
                ecolor="black",
                elinewidth=1.5,
                capsize=capsize,
                capthick=1.5,
                zorder=3      # ✅ 텍스트보다 뒤로
            )

            # ✅ 평균 텍스트: bbox + zorder 높여서 안 가리게
            ax.text(
                x_center, mean_v * 0.55,   # 막대 안쪽(필요하면 0.45~0.70 조절)
                f"{mean_v:.2f}",
                ha="center", va="center",
                fontsize=11,
                color="#333333",
                zorder=10,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7)
            )

    # (5) p-value: 각 그룹(x)에서 흡연 vs 비흡연
    #     -> 흡연자 막대 위에만 별표
    for x_i, x_label in enumerate(order if order is not None else d[x].dropna().unique()):
        smoker_vals   = d[(d[x] == x_label) & (d[hue] == "흡연자")][y].dropna()
        nonsmoke_vals = d[(d[x] == x_label) & (d[hue] == "비흡연자")][y].dropna()

        if len(smoker_vals) < 2 or len(nonsmoke_vals) < 2:
            continue

        # Welch t-test
        p = ttest_ind(smoker_vals, nonsmoke_vals, equal_var=False).pvalue
        star = star_from_p(p)
        if star == "":
            continue

        # ✅ 검증용 로그
        print(f"[{x_label}] n(흡연)={len(smoker_vals)}, n(비흡연)={len(nonsmoke_vals)} | "
              f"mean(흡연)={smoker_vals.mean():.3f}, sd(흡연)={smoker_vals.std(ddof=1):.3f} | "
              f"mean(비흡연)={nonsmoke_vals.mean():.3f}, sd(비흡연)={nonsmoke_vals.std(ddof=1):.3f} | "
              f"p={p:.4g} {star}")

        smoker_bar = bar_containers[0].patches[x_i]
        x_center = smoker_bar.get_x() + smoker_bar.get_width() / 2

        row = summary[(summary[x] == x_label) & (summary[hue] == "흡연자")]
        mean_v = float(row["mean"].iloc[0]) if not row.empty else smoker_bar.get_height()
        sd_v   = float(row["sd"].iloc[0])   if not row.empty else 0

        ax.text(
            x_center,
            mean_v + sd_v + (ax.get_ylim()[1] * 0.03),
            star,
            ha="center", va="bottom",
            fontsize=16, fontweight="bold",
            color="black",
            zorder=11
        )

    # -------------------------
    # 2) 마무리 스타일
    # -------------------------

    plt.title(title, fontweight="bold", pad=10)
    plt.ylabel(y_label)
    plt.tight_layout()
    plt.show()

In [ ]:
SMOKER_PALETTE = {"흡연자": "#F08080","비흡연자": "#B0E0E6"}

def cohen_d(x1, x2):
    """Cohen's d (pooled SD)"""
    x1 = pd.Series(x1).dropna()
    x2 = pd.Series(x2).dropna()
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return np.nan

    s1 = x1.std(ddof=1)
    s2 = x2.std(ddof=1)
    sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    if sp == 0 or np.isnan(sp):
        return np.nan
    return (x1.mean() - x2.mean()) / sp


def violin_with_median(df, x, y, title, y_label, hue=None, order=None):
    d = df.copy()

    # 혹시 order에 없는 값이 섞여 있으면 제거(라벨/정렬 안정화)
    d = d[d[x].isin(order)].copy()
    d[x] = pd.Categorical(d[x], categories=list(order), ordered=True)

    plt.figure(figsize=(12, 6))

    ax = sns.violinplot(
        data=d,
        x=x,
        y=y,
        hue=hue,
        split=True,
        inner="quartile",
        palette=SMOKER_PALETTE,
        order=list(order),
        linewidth=1
    )

    # -------------------------
    # 1) 중앙값 계산 & 표시 (좌/우 분리)
    # -------------------------
    medians = (
        d.groupby([x, hue])[y]
         .median()
         .reset_index()
    )

    for i, cat in enumerate(order):
        for g in ["흡연자", "비흡연자"]:
            v = medians[(medians[x] == cat) & (medians[hue] == g)][y]
            if v.empty:
                continue
            y_med = float(v.iloc[0])

            x_offset = -0.12 if g == "흡연자" else 0.12

            ax.text(
                i + x_offset, y_med,
                f"{y_med:.2f}",
                ha="center", va="center",
                fontsize=10,
                color="#333333",
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    fc="white",
                    ec=SMOKER_PALETTE[g],
                    alpha=0.85
                )
            )

    # -------------------------
    # 2) 각 BMI 그룹의 n(흡연/비흡연) + Cohen's d 계산 → x축 라벨에 반영
    # -------------------------
    new_xticklabels = []
    for cat in order:
        smoker_vals = d[(d[x] == cat) & (d[hue] == "흡연자")][y].dropna()
        nons_vals   = d[(d[x] == cat) & (d[hue] == "비흡연자")][y].dropna()

        n_s = len(smoker_vals)
        n_n = len(nons_vals)
        d_val = cohen_d(smoker_vals, nons_vals)

        if np.isnan(d_val):
            label = f"{cat}\n(흡연 n={n_s} / 비흡연 n={n_n})\nd=NA"
        else:
            label = f"{cat}\n(흡연 n={n_s} / 비흡연 n={n_n})\nd={d_val:.2f}"

        new_xticklabels.append(label)

    ax.set_xticklabels(new_xticklabels)

    # -------------------------
    # 3) 스타일
    # -------------------------
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(x)
    ax.set_ylabel(y_label)
    ax.grid(axis="y", linestyle=":", alpha=0.3)

    # legend 정리(흡연자/비흡연자)
    ax.legend(title=hue, frameon=True)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()